# H2/STO-3G bond-stretching VQE energy curve

Goal: extend the H2/STO-3G VQE workflow from notebook 01 into a full
H-H bond-stretching energy curve.

Pipeline, repeated at each bond distance:

1. Define molecular geometry and basis at a given H-H distance.
2. Use PySCF through Qiskit Nature to obtain the electronic-structure problem.
3. Build the second-quantized fermionic Hamiltonian.
4. Map the fermionic Hamiltonian to qubits using Jordan-Wigner.
5. Build a Hartree-Fock initial state and UCCSD ansatz.
6. Run VQE with a classical optimizer (noiseless statevector simulation, SciPy SLSQP).
7. Compare against an exact classical eigensolver reference.

This notebook reuses the same approach as `01_h2_vqe_qiskit_nature.ipynb`,
wrapped in a helper function so it can be repeated across a grid of bond distances.

In [ ]:
import importlib.metadata as metadata

packages = [
    'qiskit',
    'qiskit-nature',
    'qiskit-algorithms',
    'pyscf',
    'numpy',
    'scipy',
    'matplotlib',
    'pandas',
]

for package in packages:
    try:
        print(f'{package}: {metadata.version(package)}')
    except metadata.PackageNotFoundError:
        print(f'{package}: NOT INSTALLED')


## 1. Bond-distance grid

We sweep the H-H distance from compressed (0.4 A) through the
experimental equilibrium (0.735 A) to strongly stretched (3.0 A).
Compressing and stretching the bond changes how much static correlation
is present in the true ground state, even though the basis set and
qubit count stay fixed.

In [ ]:
bond_lengths_angstrom = [
    0.4, 0.5, 0.6, 0.7, 0.735, 0.8, 0.9, 1.0, 1.2, 1.5, 2.0, 2.5, 3.0,
]
basis = 'sto3g'

print(f'Number of bond distances: {len(bond_lengths_angstrom)}')
print(f'Range: {min(bond_lengths_angstrom)} A to {max(bond_lengths_angstrom)} A')


## 2. Helper function: one bond distance, full pipeline

This helper repeats the exact same steps as notebook 01
(build problem, fermionic Hamiltonian, Jordan-Wigner mapping,
exact diagonalization, Hartree-Fock + UCCSD ansatz, manual SciPy VQE loop)
for a single H-H distance, and returns a dictionary of the quantities
we want to track across the whole curve.

In [ ]:
import numpy as np
from scipy.optimize import minimize

from qiskit.quantum_info import Statevector
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.units import DistanceUnit
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.circuit.library import HartreeFock, UCCSD


def run_h2_sto3g_point(bond_length_angstrom, basis='sto3g'):
    """Run the full H2/STO-3G VQE pipeline at a single H-H bond distance."""
    driver = PySCFDriver(
        atom=f'H 0 0 0; H 0 0 {bond_length_angstrom}',
        unit=DistanceUnit.ANGSTROM,
        charge=0,
        spin=0,
        basis=basis,
    )
    problem = driver.run()

    fermionic_op = problem.hamiltonian.second_q_op()

    mapper = JordanWignerMapper()
    qubit_op = mapper.map(fermionic_op)

    # Exact classical reference, possible only because this is a 4-qubit toy problem.
    hamiltonian_matrix = qubit_op.to_matrix()
    electronic_eigenvalues = np.linalg.eigvalsh(hamiltonian_matrix)
    exact_electronic_energy = float(np.min(electronic_eigenvalues).real)
    exact_energy = exact_electronic_energy + float(problem.nuclear_repulsion_energy)

    initial_state = HartreeFock(
        num_spatial_orbitals=problem.num_spatial_orbitals,
        num_particles=problem.num_particles,
        qubit_mapper=mapper,
    )
    ansatz = UCCSD(
        num_spatial_orbitals=problem.num_spatial_orbitals,
        num_particles=problem.num_particles,
        qubit_mapper=mapper,
        initial_state=initial_state,
    )

    history = []

    def energy_from_parameters(parameters):
        bound_circuit = ansatz.assign_parameters(parameters, inplace=False)
        state = Statevector.from_instruction(bound_circuit)
        electronic_energy = np.real(state.expectation_value(qubit_op))
        total_energy = electronic_energy + float(problem.nuclear_repulsion_energy)
        history.append(float(total_energy))
        return float(total_energy)

    initial_point = np.zeros(ansatz.num_parameters)

    opt_result = minimize(
        energy_from_parameters,
        initial_point,
        method='SLSQP',
        options={'maxiter': 1000, 'ftol': 1e-12},
    )

    vqe_energy = float(opt_result.fun)
    abs_error = abs(vqe_energy - exact_energy)

    return {
        'bond_length_angstrom': bond_length_angstrom,
        'spin_orbitals': 2 * problem.num_spatial_orbitals,
        'qubits': qubit_op.num_qubits,
        'pauli_terms': len(qubit_op),
        'exact_energy_hartree': exact_energy,
        'vqe_energy_hartree': vqe_energy,
        'abs_error_hartree': abs_error,
        'optimized_parameters': tuple(float(p) for p in opt_result.x),
        'optimizer_success': bool(opt_result.success),
        'optimizer_message': str(opt_result.message),
        'num_function_evaluations': len(history),
    }


## 3. Look at the ansatz once, at equilibrium

The UCCSD ansatz structure (qubits, parameter count, circuit shape) is
determined by the number of spin orbitals and particles, not by the
numerical bond distance, so it is the same shape at every point on the
curve. We build and print it once at the experimental equilibrium
distance (0.735 A) purely for illustration, matching notebook 01.

In [ ]:
_driver = PySCFDriver(
    atom='H 0 0 0; H 0 0 0.735',
    unit=DistanceUnit.ANGSTROM,
    charge=0,
    spin=0,
    basis=basis,
)
_problem = _driver.run()
_mapper = JordanWignerMapper()

_initial_state = HartreeFock(
    num_spatial_orbitals=_problem.num_spatial_orbitals,
    num_particles=_problem.num_particles,
    qubit_mapper=_mapper,
)
_ansatz = UCCSD(
    num_spatial_orbitals=_problem.num_spatial_orbitals,
    num_particles=_problem.num_particles,
    qubit_mapper=_mapper,
    initial_state=_initial_state,
)

print(f'Ansatz: {_ansatz.__class__.__name__}')
print(f'Number of qubits in ansatz: {_ansatz.num_qubits}')
print(f'Number of variational parameters: {_ansatz.num_parameters}')


## 4. Run the energy curve

Now we repeat the full pipeline at every bond distance in the grid and
collect the results into a table.

In [ ]:
import pandas as pd

results = []
for bond_length_angstrom in bond_lengths_angstrom:
    print(f'Running bond distance {bond_length_angstrom} A ...')
    point_result = run_h2_sto3g_point(bond_length_angstrom, basis=basis)
    results.append(point_result)

results_df = pd.DataFrame(results)
print()
print(f'Completed {len(results_df)} bond distances.')


## 5. Results table

A compact view of the key quantities at every bond distance.

In [ ]:
display_columns = [
    'bond_length_angstrom',
    'spin_orbitals',
    'qubits',
    'pauli_terms',
    'exact_energy_hartree',
    'vqe_energy_hartree',
    'abs_error_hartree',
    'optimizer_success',
    'num_function_evaluations',
]

with pd.option_context('display.float_format', lambda v: f'{v:.10f}'):
    print(results_df[display_columns].to_string(index=False))


Optimized UCCSD parameters and optimizer messages at each bond distance:

In [ ]:
for _, row in results_df.iterrows():
    params_str = ', '.join(f'{p:.8f}' for p in row['optimized_parameters'])
    print(
        f"{row['bond_length_angstrom']:.3f} A -> "
        f"params=({params_str}), "
        f"success={row['optimizer_success']}, "
        f"message='{row['optimizer_message']}'"
    )


## 6. Plot: energy curve and VQE error

Top panel: exact diagonalization energy and VQE energy versus H-H bond
distance. Bottom panel: absolute VQE error versus bond distance, on a
log scale, since UCCSD is expected to track the exact result closely
at every point for this minimal system.

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

fig, (ax_energy, ax_error) = plt.subplots(2, 1, figsize=(8, 9), sharex=True)

ax_energy.plot(
    results_df['bond_length_angstrom'],
    results_df['exact_energy_hartree'],
    'o-', label='Exact diagonalization',
)
ax_energy.plot(
    results_df['bond_length_angstrom'],
    results_df['vqe_energy_hartree'],
    'x--', label='VQE (UCCSD)',
)
ax_energy.set_ylabel('Total energy (Hartree)')
ax_energy.set_title('H2/STO-3G energy curve: exact vs VQE')
ax_energy.legend()
ax_energy.grid(True, alpha=0.3)

ax_error.semilogy(
    results_df['bond_length_angstrom'],
    results_df['abs_error_hartree'],
    's-', color='tab:red',
)
ax_error.set_xlabel('H-H bond distance (Angstrom)')
ax_error.set_ylabel('|VQE - exact| (Hartree)')
ax_error.set_title('Absolute VQE error vs bond distance')
ax_error.grid(True, alpha=0.3, which='both')

fig.tight_layout()

figure_path = Path('figures') / 'h2_energy_curve.png'
figure_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(figure_path, dpi=150)
print(f'Saved figure to {figure_path}')

plt.show()


## 7. Scientific interpretation

- This is still a toy problem: H2/STO-3G has only 2 spatial orbitals
  (4 spin orbitals, 4 qubits) at every bond distance in this sweep.
- The point of this notebook is not chemical accuracy for H2 itself.
  The point is to observe how the same VQE workflow (Jordan-Wigner
  mapping, Hartree-Fock reference, UCCSD ansatz, statevector VQE)
  behaves as the bond is stretched.
- Bond stretching is scientifically useful because it moves the true
  ground state from near-equilibrium single-reference character
  (Hartree-Fock is a good starting point) toward stronger
  static-correlation character at large separation (Hartree-Fock
  becomes a poor starting point, and a single Slater determinant no
  longer describes the dissociated system well).
- In this specific minimal-basis case, H2/STO-3G has only one occupied
  and one virtual spatial orbital, so the only possible correlating
  excitation is the double excitation from the bonding to the
  antibonding orbital. UCCSD with this single double excitation is
  therefore equivalent to full configuration interaction (exact
  diagonalization) at every bond distance, not just at equilibrium. This
  is why the VQE error is expected to stay at the level of the
  optimizer's numerical precision across the whole curve, even as
  static correlation grows with bond distance: the ansatz has enough
  flexibility to reach the exact answer everywhere in this tiny system.
- H2/STO-3G therefore remains far too small to be a meaningful example
  of quantum advantage: the whole problem fits in a 16x16 matrix that
  is trivial to diagonalize classically, and UCCSD is essentially exact
  by construction rather than an approximation being tested.
- The next scientifically more interesting step is to move to a larger
  molecule, such as LiH or H2O, where the active space has more than
  one occupied/virtual pair. There, UCCSD becomes a genuine approximation
  to full configuration interaction rather than an exact match, and
  bond stretching can reveal real UCCSD error growth as static
  correlation increases.

## Limitations

1. This is an ideal statevector simulation, not a hardware calculation.
2. No sampling noise is included.
3. No device noise, decoherence, readout error, or transpilation constraints are included.
4. H2/STO-3G is too small to demonstrate quantum advantage at any bond distance.
5. Exact diagonalization is possible here only because the Hilbert space is tiny (4 qubits).
6. Jordan-Wigner is conceptually simple but not always the most resource-efficient mapping.
7. UCCSD happens to be exact for this specific 2-orbital system at every geometry; this will not hold for larger active spaces such as LiH or H2O.
8. The optimizer is restarted from the same all-zero initial point at each bond distance; no attempt is made to warm-start from a neighboring point on the curve.